# Bishe LLM for Genesis — Google Colab

本 Notebook 用于在 Google Colab 上运行 LLM 微调、评估与推理基准测试流水线。

**支持的 GPU**: T4 (16GB) / L4 (24GB) / A100 (40/80GB)

**核心功能**:
- 环境自动检测与配置
- Google Drive 持久化存储（模型、训练检查点、数据集）
- 支持 LoRA / QLoRA / DoRA / GaLore 四种微调方法
- 中断自动恢复（从最近检查点继续训练）

**使用方式**: 按顺序运行每个 Cell。详细说明请参阅 `CLOUD_README.md`。

## 1. 环境检测与 GPU 信息

In [ ]:
import subprocess, sys, os, shutil, platform, psutil

print("=" * 60)
print("环境检测")
print("=" * 60)

# Python 版本
print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")

# 检测 Colab
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Colab   : {'Yes' if IN_COLAB else 'No'}")

# RAM & Disk
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
disk = shutil.disk_usage("/")
print(f"RAM     : {ram_gb:.1f} GB")
print(f"Disk    : {disk.free / (1024**3):.1f} GB free / {disk.total / (1024**3):.1f} GB total")

# GPU 信息
print("\n" + "-" * 60)
print("GPU 信息")
print("-" * 60)
try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    print(result.stdout)
except Exception as e:
    print(f"[WARN] nvidia-smi 不可用: {e}")
    print("请确保已选择 GPU 运行时: 运行时 -> 更改运行时类型 -> 硬件加速器 -> GPU")

# CUDA 版本
try:
    import torch
    print(f"PyTorch CUDA : {torch.version.cuda}")
    print(f"CUDA 可用    : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_mb = torch.cuda.get_device_properties(0).total_mem / (1024 ** 2)
        print(f"GPU 型号     : {gpu_name}")
        print(f"显存         : {vram_mb:.0f} MB ({vram_mb/1024:.1f} GB)")
except ImportError:
    print("[INFO] PyTorch 尚未安装，将在后续步骤安装")

## 2. 挂载 Google Drive（持久化存储）

In [ ]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Bishe_LLM_for_genesis")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # 在 Drive 上创建持久目录
    for subdir in ["model", "output", "data_prepare"]:
        (DRIVE_PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)
    print(f"[OK] Google Drive 已挂载，持久目录: {DRIVE_PROJECT_DIR}")
else:
    print("[INFO] 非 Colab 环境，跳过 Drive 挂载")

## 3. 用户配置

根据需要修改以下参数：

In [ ]:
# ===================== 用户配置（按需修改） =====================

# Git 仓库
REPO_URL = "https://github.com/Lingalcc/Bishe_LLM_for_genesis.git"
BRANCH = "cloud-version"

# 微调方法: lora / qlora / dora / galore
#   T4  (16GB) 推荐 qlora
#   L4  (24GB) 推荐 lora / qlora / dora
#   A100 (40/80GB) 均可
FINETUNE_METHOD = "qlora"

# 基座模型 (HuggingFace model id)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# HuggingFace Token (私有模型需要，公开模型留空)
HF_TOKEN = ""  # 或使用 Colab Secrets: 从 userdata 读取

# API Keys (数据生成/评估需要，微调不需要)
DEEPSEEK_API_KEY = ""  # 用于数据生成
OPENAI_API_KEY = ""    # 用于 API 模式评估

# 是否从检查点恢复训练 (Colab 中断后设为 True)
RESUME_FROM_CHECKPOINT = False

# ===================== 配置结束 =====================

# 尝试从 Colab Secrets 读取 token/key
if IN_COLAB:
    try:
        from google.colab import userdata
        if not HF_TOKEN:
            HF_TOKEN = userdata.get("HF_TOKEN", "")
        if not DEEPSEEK_API_KEY:
            DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY", "")
        if not OPENAI_API_KEY:
            OPENAI_API_KEY = userdata.get("OPENAI_API_KEY", "")
    except Exception:
        pass

# 设置环境变量
if DEEPSEEK_API_KEY:
    os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

PROJECT_DIR = Path("/content/Bishe_LLM_for_genesis")

print(f"仓库地址       : {REPO_URL}")
print(f"分支           : {BRANCH}")
print(f"微调方法       : {FINETUNE_METHOD}")
print(f"基座模型       : {MODEL_ID}")
print(f"HF Token       : {'已设置' if HF_TOKEN else '未设置'}")
print(f"DEEPSEEK_API_KEY: {'已设置' if DEEPSEEK_API_KEY else '未设置'}")
print(f"OPENAI_API_KEY : {'已设置' if OPENAI_API_KEY else '未设置'}")
print(f"恢复训练       : {RESUME_FROM_CHECKPOINT}")

## 4. 克隆/更新项目代码

In [ ]:
if PROJECT_DIR.exists():
    print(f"[INFO] 项目已存在: {PROJECT_DIR}，执行更新...")
    !cd {PROJECT_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd {PROJECT_DIR} && git submodule update --init --recursive
else:
    print(f"[INFO] 首次克隆项目...")
    !git clone --recurse-submodules -b {BRANCH} {REPO_URL} {PROJECT_DIR}

os.chdir(str(PROJECT_DIR))
print(f"\n[OK] 工作目录: {os.getcwd()}")
!git log --oneline -5

## 5. 安装依赖

In [ ]:
%%time
os.chdir(str(PROJECT_DIR))

# 安装项目本身 (editable mode)
!pip install -e . --quiet 2>&1 | tail -5

# 安装 LlamaFactory
!pip install -e LlamaFactory --quiet 2>&1 | tail -5

# 安装微调方法额外依赖
!pip install peft bitsandbytes accelerate datasets huggingface_hub --quiet 2>&1 | tail -3

# GaLore 需要额外包
if FINETUNE_METHOD == "galore":
    !pip install galore-torch --quiet 2>&1 | tail -3

print("\n[OK] 依赖安装完成")

## 6. 验证环境

In [ ]:
import torch
import transformers
import peft
import accelerate
import datasets

print("=" * 60)
print("依赖版本检查")
print("=" * 60)
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"peft          : {peft.__version__}")
print(f"accelerate    : {accelerate.__version__}")
print(f"datasets      : {datasets.__version__}")

try:
    import bitsandbytes
    print(f"bitsandbytes  : {bitsandbytes.__version__}")
except ImportError:
    print("bitsandbytes  : [未安装] (QLoRA 需要此依赖)")

# CUDA 检查
print("\n" + "-" * 60)
assert torch.cuda.is_available(), "CUDA 不可用！请检查运行时类型是否选择了 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)
print(f"GPU           : {gpu_name}")
print(f"显存          : {vram_gb:.1f} GB")
print(f"CUDA 版本     : {torch.version.cuda}")

# 微调方法兼容性建议
print("\n" + "-" * 60)
print("微调方法与 GPU 兼容性")
print("-" * 60)
if vram_gb < 20:
    rec = "qlora (推荐), lora (可能 OOM)"
elif vram_gb < 30:
    rec = "lora (推荐), qlora, dora"
else:
    rec = "所有方法均可 (lora / qlora / dora / galore)"
print(f"当前 GPU 显存  : {vram_gb:.0f} GB")
print(f"推荐方法       : {rec}")
print(f"当前选择       : {FINETUNE_METHOD}")

if vram_gb < 20 and FINETUNE_METHOD not in ("qlora",):
    print(f"\n[WARN] 当前 GPU 显存仅 {vram_gb:.0f}GB，方法'{FINETUNE_METHOD}'可能 OOM，建议使用 qlora")

# llamafactory-cli 检查
import shutil as _shutil
if _shutil.which("llamafactory-cli"):
    print("\n[OK] llamafactory-cli 可用")
else:
    print("\n[INFO] llamafactory-cli 不在 PATH 中，将使用 python -m llamafactory.cli")

print("\n[OK] 环境验证通过")

## 7. 关联持久化存储

将 `model/`、`output/`、`data_prepare/` 通过符号链接指向 Google Drive，确保数据在运行时重置后不丢失。

In [ ]:
import os
from pathlib import Path

os.chdir(str(PROJECT_DIR))

SYMLINK_MAP = {
    "model":        DRIVE_PROJECT_DIR / "model",
    "output":       DRIVE_PROJECT_DIR / "output",
    "data_prepare": DRIVE_PROJECT_DIR / "data_prepare",
}

if IN_COLAB:
    for local_name, drive_path in SYMLINK_MAP.items():
        local_path = PROJECT_DIR / local_name
        drive_path.mkdir(parents=True, exist_ok=True)

        if local_path.is_symlink():
            # 已经是符号链接，检查目标是否正确
            if local_path.resolve() == drive_path.resolve():
                print(f"[OK] {local_name}/ -> {drive_path} (已存在)")
                continue
            else:
                local_path.unlink()

        if local_path.exists() and not local_path.is_symlink():
            # 如果本地目录已有内容，先迁移到 Drive
            import shutil
            for item in local_path.iterdir():
                dest = drive_path / item.name
                if not dest.exists():
                    if item.is_dir():
                        shutil.copytree(str(item), str(dest))
                    else:
                        shutil.copy2(str(item), str(dest))
            shutil.rmtree(str(local_path))
            print(f"[INFO] 已将 {local_name}/ 内容迁移到 Drive")

        local_path.symlink_to(drive_path)
        print(f"[OK] {local_name}/ -> {drive_path}")

    # 确保 .cache 目录也创建
    (PROJECT_DIR / ".cache").mkdir(parents=True, exist_ok=True)
    print(f"\n[OK] 持久化存储已关联，数据将保存到 Google Drive")
else:
    print("[INFO] 非 Colab 环境，使用本地目录")

## 8. 下载基座模型

In [ ]:
%%time
os.chdir(str(PROJECT_DIR))

# 将 HuggingFace model id 转换为本地目录名 (org/repo -> org_repo)
model_local_name = MODEL_ID.replace("/", "_")
model_path = PROJECT_DIR / "model" / model_local_name

if model_path.exists() and any(model_path.iterdir()):
    print(f"[OK] 模型已存在: {model_path}")
    print(f"     文件数: {sum(1 for _ in model_path.rglob('*') if _.is_file())}")
else:
    print(f"[INFO] 下载模型: {MODEL_ID}")
    print(f"[INFO] 保存到: {model_path}")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(model_path),
        local_dir_use_symlinks=False,
        token=HF_TOKEN or None,
    )
    print(f"[OK] 模型下载完成: {model_path}")

## 9. 准备数据集

- 方式 A: 使用已有数据集（如 Drive 中已有）
- 方式 B: 通过 API 生成数据集（需要 DEEPSEEK_API_KEY）
- 方式 C: 上传数据集文件到 `data_prepare/`

In [ ]:
os.chdir(str(PROJECT_DIR))

data_dir = PROJECT_DIR / "data_prepare"
alpaca_file = data_dir / "genesis_franka_toolcall_alpaca.json"
splits_dir = data_dir / "splits"
train_file = splits_dir / "train.json"

# 检查数据集是否已存在
if alpaca_file.exists():
    import json
    with open(alpaca_file) as f:
        sample_count = len(json.load(f))
    print(f"[OK] 数据集已存在: {alpaca_file}")
    print(f"     样本数: {sample_count}")

    if train_file.exists():
        print(f"[OK] 数据集分割已存在: {splits_dir}")
    else:
        print("[INFO] 执行数据集分割...")
        !python cli.py data calibrate
        from src.data_core.split_dataset import run_split_from_merged_config
        from src.utils.config import load_config
        cfg = load_config("configs/base.yaml")
        run_split_from_merged_config(cfg)
        print("[OK] 数据集分割完成")
elif DEEPSEEK_API_KEY:
    print("[INFO] 数据集不存在，通过 API 生成...")
    !python cli.py data generate --config experiments/01_data_exp/configs/api_generate.yaml
    print("[INFO] 执行数据校验...")
    !python cli.py data calibrate
    print("[OK] 数据生成完成")
else:
    if IN_COLAB:
        print("[WARN] 数据集不存在且未设置 DEEPSEEK_API_KEY")
        print("请选择以下方式之一:")
        print("  1. 将数据集文件上传到 Google Drive:")
        print(f"     {DRIVE_PROJECT_DIR / 'data_prepare' / 'genesis_franka_toolcall_alpaca.json'}")
        print("  2. 在上方配置 Cell 中设置 DEEPSEEK_API_KEY，重新运行")
        print("  3. 使用 Colab 文件上传:")
        print("     from google.colab import files; files.upload()")
    else:
        print("[WARN] 数据集不存在，请先准备数据")

## 10. 微调训练

使用 LlamaFactory 进行模型微调。训练 checkpoints 保存在 Google Drive，中断后可恢复。

In [ ]:
os.chdir(str(PROJECT_DIR))

import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024 ** 3)

print("=" * 60)
print(f"微调配置")
print("=" * 60)
print(f"方法           : {FINETUNE_METHOD}")
print(f"GPU            : {gpu_name} ({vram_gb:.0f} GB)")
print(f"恢复训练       : {RESUME_FROM_CHECKPOINT}")

# 构建训练命令
cmd_parts = [
    "python", "cli.py", "finetune", "start",
    "--finetune-method", FINETUNE_METHOD,
]

# 添加恢复训练参数
extra_args = []
if RESUME_FROM_CHECKPOINT:
    # 查找最近的 checkpoint 目录
    output_method_map = {
        "lora": "output/qwen2.5-3b-genesis-lora",
        "qlora": "output/qwen2.5-3b-genesis-qlora",
        "dora": "output/qwen2.5-3b-genesis-dora",
        "galore": "output/qwen2.5-3b-genesis-galore",
    }
    output_dir = PROJECT_DIR / output_method_map.get(FINETUNE_METHOD, f"output/qwen2.5-3b-genesis-{FINETUNE_METHOD}")
    checkpoints = sorted(output_dir.glob("checkpoint-*")) if output_dir.exists() else []
    if checkpoints:
        latest_ckpt = checkpoints[-1]
        extra_args.extend(["--", f"resume_from_checkpoint={latest_ckpt}"])
        print(f"恢复检查点     : {latest_ckpt.name}")
    else:
        print("[WARN] 未找到检查点，将从头开始训练")

cmd = " ".join(cmd_parts + extra_args)
print(f"\n执行命令: {cmd}")
print("=" * 60)

!{cmd}

## 11. 准确率评估

对微调后的模型进行 toolcall 准确率评估。

In [ ]:
os.chdir(str(PROJECT_DIR))

# 确定微调后模型路径
finetuned_model_map = {
    "lora": "model/qwen2.5-3b-genesis-lora",
    "qlora": "model/qwen2.5-3b-genesis-qlora",
    "dora": "model/qwen2.5-3b-genesis-dora",
    "galore": "model/qwen2.5-3b-genesis-galore",
}
finetuned_path = finetuned_model_map.get(FINETUNE_METHOD, f"model/qwen2.5-3b-genesis-{FINETUNE_METHOD}")

# 检查是否有待评估的模型
finetuned_full = PROJECT_DIR / finetuned_path
output_dir_map = {
    "lora": "output/qwen2.5-3b-genesis-lora",
    "qlora": "output/qwen2.5-3b-genesis-qlora",
    "dora": "output/qwen2.5-3b-genesis-dora",
    "galore": "output/qwen2.5-3b-genesis-galore",
}
output_dir = PROJECT_DIR / output_dir_map.get(FINETUNE_METHOD, f"output/qwen2.5-3b-genesis-{FINETUNE_METHOD}")

# 使用 output 目录中的模型（训练后直接评估）
eval_model_path = str(output_dir) if output_dir.exists() else str(finetuned_full)

if not Path(eval_model_path).exists():
    print(f"[WARN] 未找到微调模型: {eval_model_path}")
    print("请先运行微调步骤 (Cell 10)")
else:
    print(f"[INFO] 评估模型: {eval_model_path}")
    # 创建临时评估配置覆盖
    eval_override = f"""
test:
  accuracy_eval:
    mode: local
    model_path: {eval_model_path}
    backend: transformers
    temperature: 0.0
    max_new_tokens: 512
    num_samples: 50
    report_file: experiments/03_eval_exp/reports/accuracy_report_colab.json
"""
    eval_config_path = PROJECT_DIR / ".cache" / "colab_eval_override.yaml"
    eval_config_path.parent.mkdir(parents=True, exist_ok=True)
    eval_config_path.write_text(eval_override)

    !python cli.py eval accuracy --config {eval_config_path}

## 12. 推理性能基准测试

对比基座模型与微调模型的推理性能。

In [ ]:
os.chdir(str(PROJECT_DIR))

model_local_name = MODEL_ID.replace("/", "_")
base_model = f"model/{model_local_name}"

print("=" * 60)
print("推理性能基准测试")
print("=" * 60)

# 基座模型
if (PROJECT_DIR / base_model).exists():
    print(f"\n--- 基座模型: {base_model} ---")
    !python cli.py eval benchmark \
        --backend transformers \
        --model-path {base_model} \
        --num-samples 16 \
        --output-json experiments/02_finetune_exp/reports/benchmark_base_colab.json
else:
    print(f"[WARN] 基座模型不存在: {base_model}")

# 微调模型
if Path(eval_model_path).exists():
    print(f"\n--- 微调模型: {eval_model_path} ---")
    !python cli.py eval benchmark \
        --backend transformers \
        --model-path {eval_model_path} \
        --num-samples 16 \
        --output-json experiments/02_finetune_exp/reports/benchmark_finetuned_colab.json
else:
    print(f"[SKIP] 微调模型不存在，跳过微调模型基准测试")

## 13. 结果汇总

查看训练输出和评估报告，确认数据已保存到 Google Drive。

In [ ]:
os.chdir(str(PROJECT_DIR))
import json

print("=" * 60)
print("结果汇总")
print("=" * 60)

# 检查输出目录
print("\n--- 训练输出 ---")
output_base = PROJECT_DIR / "output"
if output_base.exists():
    for d in sorted(output_base.iterdir()):
        if d.is_dir():
            files = list(d.rglob("*"))
            size_mb = sum(f.stat().st_size for f in files if f.is_file()) / (1024**2)
            checkpoints = [f.name for f in d.iterdir() if f.is_dir() and f.name.startswith("checkpoint-")]
            print(f"  {d.name}/  ({size_mb:.0f} MB, checkpoints: {len(checkpoints)})")
else:
    print("  (无训练输出)")

# 检查评估报告
print("\n--- 评估报告 ---")
report_dirs = [
    PROJECT_DIR / "experiments" / "02_finetune_exp" / "reports",
    PROJECT_DIR / "experiments" / "03_eval_exp" / "reports",
]
for rd in report_dirs:
    if rd.exists():
        for f in sorted(rd.glob("*.json")):
            print(f"  {f.relative_to(PROJECT_DIR)}")
            try:
                data = json.loads(f.read_text())
                # 打印关键指标
                for key in ["exact_match_rate", "action_match_rate", "avg_latency", "throughput", "peak_memory"]:
                    if key in data:
                        print(f"    {key}: {data[key]}")
            except Exception:
                pass

# Google Drive 持久化状态
if IN_COLAB:
    print("\n--- Google Drive 持久化 ---")
    print(f"  Drive 项目目录: {DRIVE_PROJECT_DIR}")
    for name in ["model", "output", "data_prepare"]:
        dp = DRIVE_PROJECT_DIR / name
        if dp.exists():
            items = list(dp.iterdir())
            print(f"  {name}/ : {len(items)} 项")
    print("\n[OK] 所有数据已保存到 Google Drive，运行时重置后不会丢失")

print("\n" + "=" * 60)
print("完成！如需中断后恢复训练：")
print("  1. 在配置 Cell 中设置 RESUME_FROM_CHECKPOINT = True")
print("  2. 按顺序运行 Cell 1-7（快速恢复环境）")
print("  3. 运行 Cell 10（自动从最近检查点恢复）")
print("=" * 60)